# Seelig ortho creation
Fits an ortho on the re-preprocessed seelig scMPRA data (UMI-wise table).
Input: `/nfs/roberts/project/pi_skr2/shared/tabula_data/seelig/seelig_scmpra_umiwise.tsv.gz`

In [ ]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

In [ ]:
local = False
if local:
    cluster = LocalCluster()
    client = Client(cluster)
else:
    cluster = SLURMCluster(
        cores=8,
        memory="48G",
        processes=8,
        job_extra_directives=[
            "-p day",
            "--job-name=seelig_ortho_worker",
            "--time=6:00:00",
            "--output=worker_%j.out"
        ]
    )
    cluster.scale(jobs=3)
    client = Client(
        cluster,
        timeout=f"{5*60}s",
        heartbeat_interval="20s"
    )

print(client.dashboard_link)

In [ ]:
data_root = Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
path = data_root / "seelig"
name = "seelig_ortho_20260320"

In [ ]:
if (path / name).is_dir():
    print("[+] Model found. Loading...")
    primordial = scm.ortho.load(client, path, name)
else:
    print("[+] Model not found. Creating...")

    seelig = scm.scMPRA_data.from_tsv(str(path / "seelig_scmpra_umiwise.tsv.gz"))

    seelig.set_negative_controls([
        "AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACTCATGCGTTACGCGCCTCCGAGTTATGGGGGGGGAGGCGCGTATCTCGTGGAGAAGAAGCGATGTAACGCTTGGGCGATAAGCTTATAAGGAAGATATTT",
        "CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAGCGTATCTTATCTTCAGATGGGGATGTCGCGCATCCACCCAGTGGGCACCGCCGCTATAGAAGGGTGATAACGCTTCTCAGCCTTCAGGCTCTGGGTCTT"
    ])
    seelig.set_reference_cell("HepG2")
    seelig.ortho_filter()

    # Seelig has no transfection reporter — all unobserved (cell, CRE) combos are true zeroes
    seelig.set_consider_missing(enabled=True)

    primordial = scm.ortho()
    primordial.criss_cross(client=client, dat=seelig)
    primordial.extract_params(client)
    primordial.save(path, name)
    print("[+] Done.")

In [ ]:
client.close()
cluster.close()